# آموزشِ صدا — نسخهٔ Kaggle

**فرقش با Colab:** اینجا اجرا روی سرورِ Kaggle در پس‌زمینه می‌رود.
مرورگر را می‌بندید، کامپیوتر را خاموش می‌کنید، اینترنتتان قطع می‌شود —
هیچ‌کدام اثری ندارد.

## کاری که باید بکنید (یک بار)

۱. در پنلِ راست: `Session options`
   - **Accelerator** ← `GPU T4 x2`
   - **Internet** ← `On`
     (اگر خاموش بود، حسابتان باید با شمارهٔ موبایل تأیید شده باشد:
     `Settings` ← `Phone Verification`)
۲. سلولِ «تنظیمات» را نگاه کنید
۳. دکمهٔ **`Save Version`** بالا سمتِ راست ←
   **`Save & Run All (Commit)`** ← `Save`

**یک بار بزنید، نه دو بار.** اگر وقتی یک نسخه در حالِ اجراست دوباره
`Save` بزنید، نسخهٔ قبلی لغو می‌شود و همان پیامِ «Canceled by backend»
را می‌گیرد. بعد از زدن، از منوی سه‌نقطه ← `Versions` ببینید چند نسخه
در فهرست است.

حالا می‌توانید تب را ببندید.

## بعدش

یکی-دو ساعت بعد به همان صفحه برگردید و از تبِ `Output` دو فایل را
دانلود کنید. بعد در درایو، پوشهٔ `voice-models` بگذاریدشان.

## اگر «Exit code: 137» گرفت

۱۳۷ یعنی سیستم فرایند را کُشت (`SIGKILL`) — از بیرونِ فرایند هیچ
دلیلی در آن نیست، نه برای شما و نه برای من. دو چیز آن را می‌سازد:
نسخه‌ای که نسخهٔ بعدی لغوش کرده، و تمام‌شدنِ حافظه.

پس این نوت‌بوک بعدِ هر قدمِ سنگین رم و دیسک را چاپ می‌کند. اگر باز
کشته شد، **آخرین خطِ لاگ** جواب را دارد: اگر رم نزدیکِ سقف بود مسئله
حافظه است، و اگر نه، نسخه لغو شده.

## اگر نصفه ماند و باید ادامه بدهید

خروجیِ هر اجرای کامل یک پوشهٔ `resume/` هم دارد. در اجرای بعدی
`Add Input` ← `Your Work` ← همین نوت‌بوک را اضافه کنید؛ آموزش از
همان‌جا ادامه می‌دهد و از صفر شروع نمی‌کند.

---

منطقِ قدم‌ها در مخزن است و همین نوت‌بوک از آنجا می‌گیردش — همان کدی
که نسخهٔ Colab هم اجرا می‌کند.


## ۱ — کارتِ گرافیک، اینترنت، و جا

هر سه باید باشند، وگرنه بقیه بی‌معنی است.


In [ ]:
import subprocess, sys, os, shutil, json, time, glob

T0 = time.time()

# ══ چرا خطِ به خط ══
# وقتی خروجی به فایل می‌رود (و در Kaggle می‌رود)، پایتون آن را بلوکی
# نگه می‌دارد. یعنی اگر فرایند با SIGKILL بمیرد، هرچه در بافر مانده
# **هیچ‌وقت نوشته نمی‌شود** — و لاگ روی آخرین خطی می‌ایستد که اتفاقاً
# بافر را پر کرده بود، نه آخرین کاری که واقعاً انجام شده. اجرای اولِ
# Kaggle دقیقاً همین شکل بود: لاگ روی «وزن‌ها آماده‌اند» تمام شده بود
# و بعدش هیچ. این یک خط یعنی هر خط همان لحظه روی دیسک است.
try:
    sys.stdout.reconfigure(line_buffering=True)
except (AttributeError, ValueError):
    pass


def res(tag=''):
    """رم و دیسکِ همین لحظه — بعدِ هر قدمِ سنگین.

    اجرای اولِ Kaggle با «Canceled by backend. Exit code: 137» مُرد و
    لاگ چیزی نداشت: ۱۳۷ یعنی SIGKILL، و کُشنده هیچ‌وقت خودش را معرفی
    نمی‌کند. تنها راهِ فهمیدنش این است که پیش از مرگ خودمان عدد
    گذاشته باشیم — همان درسِ «گزارشی که خوانده نمی‌شود».
    """
    m = {}
    for ln in open('/proc/meminfo'):
        p = ln.split()
        if len(p) > 1:
            m[p[0].rstrip(':')] = int(p[1]) / 1048576.0
    used = m.get('MemTotal', 0) - m.get('MemAvailable', 0)
    where = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/'
    d = shutil.disk_usage(where)
    print('   [%5.0f ثانیه]  رم %.1f از %.1f گیگ  ·  دیسکِ آزاد %.0f گیگ   %s'
          % (time.time() - T0, used, m.get('MemTotal', 0),
             d.free / float(1 << 30), tag), flush=True)


g = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True)
name = (g.stdout or b'').decode().strip()
if g.returncode != 0 or not name:
    raise SystemExit('GPU روشن نیست: Session options ← Accelerator ← GPU T4')
print('کارت:', name.splitlines()[0])

# اینترنت را همین‌جا می‌سنجیم، نه سه سلول بعد وقتی دانلود می‌شکند.
r = subprocess.run(['curl', '-sSfI', '-m', '20',
                    'https://raw.githubusercontent.com'],
                   capture_output=True)
if r.returncode:
    raise SystemExit('اینترنت خاموش است: Session options ← Internet ← On')
print('اینترنت: وصل')
res('آغاز')

## ۲ — تنظیمات


In [ ]:
VOICE = 'razavi'

DRIVE_IDS = [
    '1YRI2p7Qv3hh2dcNPMZDmbNUel0XCYWKX',
    '1cBUasKKB2Q5JjLfpZfyjBC7ZNo72KAiD',
    '1izlhA9PRU0VWcmL-Gw7lFaW2LJ3nLUKv',
    '1QdJzUi8sk5LhuUqjHeCi9Kb4UgRYq4P5',
]

SR      = '40k'
EPOCHS  = 150
BATCH   = 8
SAVE_EVERY = 25

# در Kaggle خروجی از این پوشه برداشته می‌شود.
OUT  = '/kaggle/working'
WORK = OUT + '/work'

# پوشه‌ای که اجرای بعدی از آن ادامه می‌دهد. اگر خروجیِ یک اجرای قبلی
# را به‌عنوانِ ورودی اضافه کرده باشید، همین‌جا پیدا می‌شود.
RESUME = OUT + '/resume'

## ۳ — کد


In [ ]:
os.chdir(OUT)
if not os.path.isdir(OUT + '/rvc'):
    subprocess.run(['git', 'clone', '--depth', '1', '-q',
                    'https://github.com/RVC-Project/'
                    'Retrieval-based-Voice-Conversion-WebUI',
                    OUT + '/rvc'], check=True)

RAW = 'https://raw.githubusercontent.com/mahdighandi1989/Content-Engine/main/tools'
for f in ('rvcpipe.py', 'dsprep.py'):
    subprocess.run(['curl', '-sSLf', RAW + '/' + f,
                    '-o', OUT + '/' + f], check=True)
sys.path.insert(0, OUT)
import rvcpipe as P, dsprep as D

ROOT = OUT + '/rvc'
os.makedirs(WORK, exist_ok=True)
print('قدم‌ها:', [n for n, _ in P.steps(VOICE, '/x', ROOT, sr=SR)])
res('کد')

## ۴ — وابستگی‌ها


In [ ]:
deps = P.TRAIN_DEPS + D.DS_DEPS + ['gdown']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q']
               + deps, check=True)
print('نصب شد:', len(deps), 'بسته')
res('وابستگی‌ها')

## ۵ — وزن‌های پایه

حدود ۵۵۰ مگابایت. پروانه‌ها سنجیده شده: کدِ RVC ‏MIT · وزن‌ها ‏MIT ·
ContentVec ‏MIT · RMVPE ‏Apache-2.0 · مدلِ گوینده ‏Apache-2.0.


In [ ]:
os.chdir(ROOT)
hf = shutil.which('hf') or shutil.which('huggingface-cli')
for cmd in P.assetCmds_(py=sys.executable, sr=SR, hf=hf or 'hf'):
    if cmd[:3] == [sys.executable, '-m', 'pip']:
        continue
    if cmd[0] in ('hf', 'huggingface-cli'):
        cmd[0] = hf or cmd[0]
    r = subprocess.run(cmd)
    if r.returncode:
        raise SystemExit('دانلود ناموفق: ' + ' '.join(cmd[:4]))
print('وزن‌ها آماده‌اند')
res('وزن‌ها')

## ۶ — گرفتنِ ضبط‌ها

هر فایل جدا اعلام می‌شود. قدمی که چیزی چاپ نمی‌کند، وقتی کشته شود
معلوم نیست کجا بوده — اجرای اول دقیقاً همین‌جا گم شد.


In [ ]:
import gdown
RAWDIR = WORK + '/raw'
os.makedirs(RAWDIR, exist_ok=True)
srcs = []
for i, fid in enumerate(DRIVE_IDS):
    dst = os.path.join(RAWDIR, 'in%d' % (i + 1))
    if not (os.path.exists(dst) and os.path.getsize(dst) > 100000):
        print('  دانلودِ %d از %d …' % (i + 1, len(DRIVE_IDS)), flush=True)
        gdown.download(id=fid, output=dst, quiet=True)
    if os.path.exists(dst) and os.path.getsize(dst) > 100000:
        srcs.append(dst)
        print('  %d: %.0f مگابایت' % (i + 1,
              os.path.getsize(dst) / 1048576.0), flush=True)
    else:
        print('  %d: نیامد (دسترسی؟) %s' % (i + 1, fid), flush=True)
print('%d از %d ضبط آماده است' % (len(srcs), len(DRIVE_IDS)))
res('ضبط‌ها')
if not srcs:
    raise SystemExit('هیچ فایلی نیامد — اشتراکِ فایل‌ها باید «هر کسی با لینک» باشد')

## ۷ — جداکردنِ موسیقی و ساختِ دیتاست

سه دروازه: فاصله‌های بلند، کفِ هر تکه، و شباهت به گویندهٔ غالب.
روی چهار ضبطِ بلند چند دقیقه طول می‌کشد، پس هر فایل به‌محضِ تمام‌شدن
گزارش می‌دهد.


In [ ]:
DS = WORK + '/dataset'
os.makedirs(DS, exist_ok=True)
have = [f for f in os.listdir(DS) if f.endswith('.wav')]
if have:
    print('دیتاست از پیش آماده است: %d تکه' % len(have))
else:
    def onFile(rows):
        r_ = rows[-1]
        print('  %-22s %7.1f ثانیه → %3d تکه'
              % (r_['file'][:22], r_['seconds'], r_.get('segments', 0)),
              flush=True)
        res('تحلیلِ %d از %d' % (len(rows), len(srcs)))

    segs, rep = D.buildDataset_(srcs, DS, sampleDir=OUT, onFile=onFile)
    print('\n' + rep['line'])
    res('دیتاست')
    if not segs:
        raise SystemExit('هیچ تکه‌ای نماند — ضبط‌ها را ببینید')

## ۸ — ادامه از اجرای پیشین (اگر باشد)

اگر خروجیِ یک اجرای قبلی را به‌عنوانِ ورودی اضافه کرده باشید،
چک‌پوینت‌هایش برداشته می‌شود و آموزش از همان‌جا ادامه می‌دهد؛
`train.py` خودش آخرین چک‌پوینت را پیدا می‌کند.

استخراج دوباره اجرا می‌شود — چند دقیقه است و نتیجه‌اش قطعی است.
آنچه ارزشِ حمل دارد ساعت‌های آموزش است، نه آن.


In [ ]:
dst = os.path.join(ROOT, 'logs', VOICE)
os.makedirs(dst, exist_ok=True)
found = sorted(glob.glob('/kaggle/input/*/resume/' + VOICE + '/*.pth'))
for p in found:
    shutil.copy(p, dst)
    print('  برداشته شد: %s (%.0f مگابایت)'
          % (os.path.basename(p), os.path.getsize(p) / 1048576.0))
print('ادامه از %d چک‌پوینت' % len(found) if found
      else 'اجرای تازه — چک‌پوینتی از پیش نیست')

## ۹ — آموزش


In [ ]:
P.preLog_(ROOT, VOICE)
env = P.env(ROOT)
steps = P.steps(VOICE, DS, ROOT, sr=SR, f0method='rmvpe', epochs=EPOCHS,
                save_every=SAVE_EVERY, version='v2', gpus='0', n_p=2,
                batch=BATCH, py=sys.executable, latest=1)
done = os.path.join(ROOT, 'logs', VOICE, '3_feature768')
for nm, cmd in steps:
    if nm in ('preprocess', 'extract_f0', 'extract_feature') \
            and os.path.isdir(done) and os.listdir(done):
        print('%s: از پیش انجام شده' % nm)
        continue
    if nm == 'train':
        info = P.preTrain_(ROOT, VOICE, sr=SR, version='v2')
        print('فهرستِ آموزش:', info)
        if not info['from_dataset']:
            raise SystemExit('فهرست خالی است — استخراج چیزی نساخت')
    t0 = time.time()
    print('\n=== %s ===' % nm, flush=True)
    r = subprocess.run(cmd, cwd=ROOT, env=env)
    print('%s: %ds' % (nm, time.time() - t0))
    res(nm)
    if r.returncode:
        raise SystemExit('قدمِ «%s» شکست خورد (کد %d)' % (nm, r.returncode))

## ۱۰ — نتیجه

دو فایل در `/kaggle/working` می‌نشیند و در تبِ **Output** قابلِ دانلود
است. همین دو تا چیزی است که موتور به کار می‌برد.

کنارشان پوشهٔ `resume/` می‌ماند تا اجرای بعدی — اگر خواستید بیشتر
آموزش بدهید — از صفر شروع نکند. پوشهٔ کار پاک می‌شود تا خروجی سبک
بماند.


In [ ]:
o = P.outputs(VOICE, ROOT)
if not os.path.exists(o['model']):
    raise SystemExit('آموزش تمام شد ولی مدلی ساخته نشد: ' + o['model'])
saved = [shutil.copy(o['model'], OUT)]
for f in sorted(os.listdir(o['index_dir'])):
    if f.endswith('.index'):
        saved.append(shutil.copy(os.path.join(o['index_dir'], f), OUT))
for s in saved:
    print('%8.1f مگابایت  %s' % (os.path.getsize(s) / 1048576.0, s))

# ── توشهٔ اجرای بعدی ──
# فقط چک‌پوینت‌ها. ویژگی‌های استخراج‌شده چند گیگ‌اند و در چند دقیقه
# دوباره ساخته می‌شوند؛ آموزش نه.
keep = os.path.join(RESUME, VOICE)
os.makedirs(keep, exist_ok=True)
for f in sorted(glob.glob(os.path.join(ROOT, 'logs', VOICE, '*.pth'))):
    shutil.copy(f, keep)
    print('  توشه: %s (%.0f مگابایت)'
          % (os.path.basename(f), os.path.getsize(f) / 1048576.0))

shutil.rmtree(WORK, ignore_errors=True)
shutil.rmtree(ROOT, ignore_errors=True)
res('پایان')
print('\nتمام شد. از تبِ Output برشان دارید.')